# HealthLink Offline Agent Lab

Use this notebook to test each agent modularly with custom inputs.

Run cells top to bottom.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'scripts' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT =', ROOT)

In [ ]:
from config.settings import get_settings
from core.database import get_db_manager, seed_doctors
from agents.symptom_agent import symptom_agent
from agents.doctor_agent import doctor_agent
from agents.scheduling_agent import scheduling_agent
from agents.summary_agent import summary_agent
from core.schemas import SymptomExtraction, Symptom, DoctorRecommendation, Doctor, SchedulingRecommendation, TimeSlot

settings = get_settings()
print('OFFLINE_MODE =', settings.offline_mode)
if not settings.offline_mode:
    raise RuntimeError('Set OFFLINE_MODE=true in .env before running this notebook.')

db_manager = get_db_manager(settings)
print('Settings + DB ready')

In [ ]:
# Optional: seed doctors from CSV
import pandas as pd

doctors_csv = ROOT / 'data' / 'doctors.csv'
if doctors_csv.exists():
    doctors_data = pd.read_csv(doctors_csv).to_dict('records')
    with db_manager.session_scope() as session:
        seed_doctors(session, doctors_data)
    print('Doctors seeded or already present')
else:
    print('doctors.csv not found')

## 1) Symptom Agent

In [ ]:
user_input = 'I have severe headache and fever for 3 days'
symptom_result = symptom_agent(user_input=user_input, settings=settings, use_rag=False)
print(symptom_result.model_dump_json(indent=2))

## 2) Doctor Agent
Uses `symptom_result` from previous cell.

In [ ]:
with db_manager.session_scope() as session:
    doctor_result = doctor_agent(symptom_analysis=symptom_result, db_session=session, settings=settings)
print(doctor_result.model_dump_json(indent=2))

## 3) Scheduling Agent
Uses `doctor_result` + urgency from `symptom_result`.

In [ ]:
scheduling_result = scheduling_agent(
    doctor_recommendation=doctor_result,
    urgency_level=symptom_result.urgency_level,
    settings=settings,
)
print(scheduling_result.model_dump_json(indent=2))

## 4) Summary Agent
Uses all previous outputs.

In [ ]:
summary_result = summary_agent(
    symptom_analysis=symptom_result,
    doctor_recommendation=doctor_result,
    scheduling_recommendation=scheduling_result,
    settings=settings,
)
print(summary_result.model_dump_json(indent=2))

## Modular Testing With Custom Inputs
You can create any object and test one agent in isolation.

In [ ]:
# Example: run Scheduling Agent directly with custom doctor data
custom_doctor = Doctor(
    name='Dr. Test',
    specialty='Neurology',
    experience_years=12,
    rating=4.7,
    availability='Mon-Fri',
    location='Clinic A',
)
custom_doc_rec = DoctorRecommendation(
    recommended_doctors=[custom_doctor],
    specialty_rationale='Manual test input',
    match_score=0.9,
)
custom_sched = scheduling_agent(custom_doc_rec, urgency_level='high', settings=settings)
print(custom_sched.model_dump_json(indent=2))